# 四、准备neo4j文件

1.下面我要准备neo4j的关系文件和节点文件了，节点文件我准备先测试时采用属性的子集：
以users_test.csv为例，转化后保存到/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload路径下， 
我想保存的属性有： id(用户唯一标识符)、badges（用户获得的徽章列表）、banned（bool值。用户是否被禁止使用平台）、
bio（用户的个人简介或自我描述）、posts（用户发布帖子的数量）、comments（用户发布评论的数量）、user_followers(用户的粉丝数量)、user_following(用户关注的人数)、
username(用户名)， 保留这些后，并使csv文件数据结构满足neo4j neo4j-admin import导入需要

In [18]:
import csv
import ast
# 输入和输出文件路径
# datadir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/'
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
input_file = datadir + 'user_valid.csv'  # 原始文件路径
output_file = datadir + 'neo4j_data/user.csv'  # 输出文件路径

# 打开原始CSV文件读取数据
with open(input_file, 'r', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    
    # 输出文件
    with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
        # 定义需要保留的字段以及格式
        fieldnames = ['id:ID', 'badges', 'banned', 'bio', 'posts', 'comments', 'user_followers', 'user_following', 'username']
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        
        # 写入表头
        writer.writeheader()

        # 处理每一行数据
        for row in reader:
            processed_row = {}

            # id: 用户唯一标识符
            processed_row['id:ID'] = row['id:ID']

            # badges: 处理为逗号分隔字符串
            badges = row['badges']
            if badges:
                try:
                    badges_list = ast.literal_eval(badges)  # 解析字符串为列表
                    processed_row['badges'] = ",".join(map(str, badges_list))  # 转换为逗号分隔的字符串
                except Exception:
                    processed_row['badges'] = ""  # 如果解析失败，保留空
            else:
                processed_row['badges'] = ""

            # banned: 转换为布尔值（False 或 True）
            processed_row['banned'] = row['banned'] == 'True'

            # bio: 直接从原始文件中获取
            processed_row['bio'] = row['bio'] if row['bio'] else ""

            # posts: 处理为数字类型
            processed_row['posts'] = int(float(row['posts'])) if row['posts'] else 0

            # comments: 处理为数字类型
            processed_row['comments'] = int(float(row['comments'])) if row['comments'] else 0

            # user_followers: 处理为数字类型
            processed_row['user_followers'] = int(float(row['user_followers'])) if row['user_followers'] else 0

            # user_following: 处理为数字类型
            processed_row['user_following'] = int(float(row['user_following'])) if row['user_following'] else 0

            # username: 直接从原始文件中获取
            processed_row['username'] = row['username'] if row['username'] else ""

            # 写入处理后的行
            writer.writerow(processed_row)

print("数据转换完成并保存为:", output_file)


数据转换完成并保存为: /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/user.csv


2.下面是我想保留的属性：
id(评论的唯一标识符)、comments:(当前post的回复数量)、body（post的文本内容）、createdAt(Post创建时间)、creator(post发布者的唯一标识符，和user文件中的id一一对应)、score（贴纸得分）、sensitive（布尔值，post是否包含敏感内容）、upvotes(收到的点赞数量)、username（发布者的用户名）

In [19]:
import csv
from tqdm import tqdm  # 导入 tqdm 库

# 输入和输出文件路径
# datadir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/'
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
input_file = datadir + 'post_valid.csv'  # 原始文件路径
output_file = datadir + 'neo4j_data/post.csv'  # 输出文件路径


# 打开原始CSV文件读取数据
with open(input_file, 'r', encoding='utf-8') as infile:
    reader = csv.DictReader(infile)
    total_rows = sum(1 for _ in infile)  # 获取文件的总行数

    # 重新打开文件以便重新读取
    infile.seek(0)
    
    # 输出文件
    with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
        # 定义需要保留的字段以及格式
        fieldnames = ['id:ID', 'comments', 'body', 'createdAt', 'creator', 'score', 'sensitive', 'upvotes', 'username','ML1_oracle1_probability','ML1_proxy4b1_probability','ML1_proxy2b1_probability']
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        
        # 写入表头
        writer.writeheader()

        # 使用 tqdm 来为循环添加进度条
        for row in tqdm(reader, total=total_rows, desc="Processing rows"):
            processed_row = {}

            # id: 评论的唯一标识符
            processed_row['id:ID'] = row['id:ID']

            # comments: 当前post的回复数量
            processed_row['comments'] = int(float(row['comments'])) if row['comments'] else 0

            # body: post的文本内容
            processed_row['body'] = row['body'] if row['body'] else ""

            # createdAt: post的创建时间
            processed_row['createdAt'] = row['createdAt'] if row['createdAt'] else ""

            # creator: 发布者的唯一标识符
            processed_row['creator'] = row['creator']

            # score: 贴纸得分
            processed_row['score'] = float(row['score']) if row['score'] else 0.0

            # sensitive: 布尔值，post是否包含敏感内容
            processed_row['sensitive'] = row['sensitive'] == 'True'

            # upvotes: 收到的点赞数量
            processed_row['upvotes'] = int(float(row['upvotes'])) if row['upvotes'] else 0

            # username: 发布者的用户名
            processed_row['username'] = row['username'] if row['username'] else ""
            # Oracle1: ML1_Oracle1模型输出
            processed_row['ML1_oracle1_probability'] = float(row['ML1_oracle1_probability']) if row['ML1_oracle1_probability'] else 0.0
            
            # ML1_proxy4b1_probability: ML1_proxy4b1_probability模型输出
            processed_row['ML1_proxy4b1_probability'] = float(row['ML1_proxy4b1_probability']) if row['ML1_proxy4b1_probability'] else 0.0
            
            # ML1_proxy2b1_probability: distilbert-base-uncased模型输出
            processed_row['ML1_proxy2b1_probability'] = float(row['ML1_proxy2b1_probability']) if row['ML1_proxy2b1_probability'] else 0.0
            # 写入处理后的行
            writer.writerow(processed_row)

print("数据转换完成并保存为:", output_file)


Processing rows: 100%|█████████▉| 24082/24083 [00:01<00:00, 14212.63it/s]

数据转换完成并保存为: /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/post.csv


3.下面是我想保留的属性：
id(评论的唯一标识符)、comments:(当前comment的回复数量)、body（comment的文本内容）、createdAt(comment创建时间)、creator(comment发布者的唯一标识符，和user文件中的id一一对应)、score（comment得分）、sensitive（布尔值，comment是否包含敏感内容）、upvotes(收到的点赞数量)、username（发布者的用户名） 、controversy(争议指数)、parent（当前评论父级评论id）、post（评论所关联post的唯一标识符、与上面posts_test_large中post 的id一一对应）

In [20]:
import pandas as pd
from tqdm import tqdm

# 输入和输出文件路径
datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
input_file = datadir + 'comment_valid.csv'  # 原始文件路径
output_file = datadir + 'neo4j_data/comment.csv'  # 输出文件路径


# 使用 Pandas 读取 CSV 文件
df = pd.read_csv(input_file)

# 定义需要保留的列
columns_to_keep = ['id:ID', 'comments', 'body', 'createdAt', 'creator', 'score', 'sensitive', 'upvotes', 'username', 'controversy', 'parent', 'post','ML2_oracle2_probability','ML2_proxy2d2_probability','ML2_proxy4d2_probability']

# 保留指定的列并进行数据转换
df_processed = df[columns_to_keep].copy()

# 转换数据类型并调整字段以符合 Neo4j 导入要求
df_processed['id:ID'] = df_processed['id:ID'].astype(str)
df_processed['comments'] = df_processed['comments'].fillna(0).astype(int)
df_processed['body'] = df_processed['body'].fillna('')
df_processed['createdAt'] = df_processed['createdAt'].fillna('')
df_processed['creator'] = df_processed['creator'].astype(str)
df_processed['score'] = df_processed['score'].fillna(0.0).astype(float)
df_processed['sensitive'] = df_processed['sensitive'] == 'True'  # 将字符串 'True' 转为布尔值
df_processed['upvotes'] = df_processed['upvotes'].fillna(0).astype(int)
df_processed['username'] = df_processed['username'].fillna('')
df_processed['controversy'] = df_processed['controversy'].fillna(0.0).astype(float)
df_processed['parent'] = df_processed['parent'].fillna('').astype(str)
df_processed['post'] = df_processed['post'].astype(str)
# 概率列转换（关键修复）
df_processed['ML2_oracle2_probability'] = pd.to_numeric(df_processed['ML2_oracle2_probability'], errors='coerce').fillna(0.0)
df_processed['ML2_proxy2d2_probability'] = pd.to_numeric(df_processed['ML2_proxy2d2_probability'], errors='coerce').fillna(0.0)
df_processed['ML2_proxy4d2_probability'] = pd.to_numeric(df_processed['ML2_proxy4d2_probability'], errors='coerce').fillna(0.0)

# 在导入到 Neo4j 时需要将字段名称调整为符合 Neo4j 规范
# 1. 将 id 设置为 :ID，creator 设置为 :ID，确保它们符合 Neo4j 的导入要求
df_processed = df_processed.rename(columns={
    'id:ID': 'id:ID',
    'creator': 'creator',
    'post': 'post',  # 为了确保 post 作为外键在 Neo4j 中能识别
})

# 使用 tqdm 显示处理进度条
with tqdm(total=len(df_processed), desc="Processing comments") as pbar:
    # 将处理后的数据保存为新的 CSV 文件
    df_processed.to_csv(output_file, index=False)

    # 更新进度条
    pbar.update(len(df_processed))

print("数据转换完成并保存为:", output_file)


Processing comments: 100%|██████████| 110264/110264 [00:03<00:00, 33439.53it/s]

数据转换完成并保存为: /home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/comment.csv


4.comments_processed.csv 文件中的元组数量: 128588
posts_processed.csv 文件中的元组数量: 8906
users_processed.csv 文件中的元组数量: 42448

In [6]:
import pandas as pd

# 文件路径
# neo4j_datadir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/'
neo4j_datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/'
comments_file = neo4j_datadir + 'comment.csv'
posts_file = neo4j_datadir + 'post.csv'
users_file = neo4j_datadir + 'user.csv'

# 读取每个文件并计算行数
comments_df = pd.read_csv(comments_file)
posts_df = pd.read_csv(posts_file)
users_df = pd.read_csv(users_file)

# 获取每个 DataFrame 的行数（即元组数量）
comments_count = len(comments_df)
posts_count = len(posts_df)
users_count = len(users_df)

# 输出每个文件中的元组数量
print(f"comments_processed.csv 文件中的元组数量: {comments_count}")
print(f"posts_processed.csv 文件中的元组数量: {posts_count}")
print(f"users_processed.csv 文件中的元组数量: {users_count}")


comments_processed.csv 文件中的元组数量: 110264
posts_processed.csv 文件中的元组数量: 24082
users_processed.csv 文件中的元组数量: 41342


5.（1）已知comments文件中元组通过外键post和posts文件的id相连接 ，也就是说commnet节点和post几点之间通过commnet外键post和post的id连接， 建立从post到commnet的边标签为'have'
（2）posts 和comments文件中元组通过外键creator与users的id连接，建立user到post 和（3）commnet的边，边标签为‘public’, 
如果一个post和commnet存在一条边，该边标签为'have',commnet和user存在一条边，边标签为‘public’，那么要建立一条从user到post的边，边标签为view,
如何实现上面这些关系的建立，给出关系CSV创建代码，然后给出使用 neo4j neo4j-admin import导入这些CSV到neo4j中建立图，

In [12]:
import pandas as pd
import csv
from tqdm import tqdm  # 导入进度条模块

# 文件路径
neo4j_datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/'
comments_file = neo4j_datadir + 'comment.csv'
posts_file = neo4j_datadir + 'post.csv'
users_file = neo4j_datadir + 'user.csv'

# 读取 CSV 文件
comments_df = pd.read_csv(comments_file)
posts_df = pd.read_csv(posts_file)
users_df = pd.read_csv(users_file)

# 输出关系 CSV 文件路径
have_relationship_file = neo4j_datadir + 'have_relationship.csv'
public_relationship_file = neo4j_datadir +'public_relationship.csv'
view_relationship_file = neo4j_datadir +'view_relationship.csv'

# 创建关系：Post 到 Comment（have 关系）
with open(have_relationship_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([':START_ID', ':END_ID', ':TYPE'])  # 表头
    for _, comment in tqdm(comments_df.iterrows(), total=comments_df.shape[0], desc="Processing 'have' relationships"):
        post_id = comment['post']
        comment_id = comment['id:ID']
        writer.writerow([post_id, comment_id, 'have'])

# 创建关系：User 到 Post 和 Comment（public 关系）
with open(public_relationship_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([':START_ID', ':END_ID', ':TYPE'])  # 表头
    for _, comment in tqdm(comments_df.iterrows(), total=comments_df.shape[0], desc="Processing 'public' relationships for comments"):
        user_id = comment['creator']
        comment_id = comment['id:ID']
        writer.writerow([user_id, comment_id, 'public'])

    for _, post in tqdm(posts_df.iterrows(), total=posts_df.shape[0], desc="Processing 'public' relationships for posts"):
        user_id = post['creator']  # 修改为 'creator:ID'
        post_id = post['id:ID']
        writer.writerow([user_id, post_id, 'public'])

# # 创建关系：User 到 Post（view 关系）
# with open(view_relationship_file, 'w', newline='', encoding='utf-8') as f:
#     writer = csv.writer(f)
#     writer.writerow([':START_ID', ':END_ID', ':TYPE'])  # 表头
#     for _, post in tqdm(posts_df.iterrows(), total=posts_df.shape[0], desc="Processing 'view' relationships for posts"):
#         user_id = post['creator']  # 修改为 'creator:ID'
#         post_id = post['id:ID']
#         writer.writerow([user_id, post_id, 'view'])
# print("关系 CSV 文件已生成")


Processing 'public' relationships for posts: 100%|██████████| 24082/24082 [00:02<00:00, 11553.41it/s]


让在post下发表评论的用户到post也有一条view边，所有相同标签的边不能重复

In [15]:
# 创建关系：User 到 Post（view 关系）
with open(view_relationship_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow([':START_ID', ':END_ID', ':TYPE'])  # 表头
    
    view_edges = set()
    
    # 从评论中找出用户-帖子关系
    for _, comment in tqdm(comments_df.iterrows(), total=comments_df.shape[0], desc="Processing 'view' relationships from comments"):
        user_id = comment['creator']
        post_id = comment['post']
        edge = (user_id, post_id)
        if edge not in view_edges:
            view_edges.add(edge)
            writer.writerow([user_id, post_id, 'view'])
    
    # 发帖人自己也算看过帖子（可选）
    for _, post in tqdm(posts_df.iterrows(), total=posts_df.shape[0], desc="Processing 'view' relationships from posts"):
        user_id = post['creator']
        post_id = post['id:ID']
        edge = (user_id, post_id)
        if edge not in view_edges:
            view_edges.add(edge)
            writer.writerow([user_id, post_id, 'view'])


Processing 'view' relationships from posts: 100%|██████████| 24082/24082 [00:02<00:00, 11588.09it/s]


 6. 给user、post、comment添加标签，标签为：':LABEL'，分别为User、Post、Comment

In [21]:
import pandas as pd

# 文件路径
# neo4j_datadir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j_data/'
neo4j_datadir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/'
comment_file = neo4j_datadir + 'comment.csv'
post_file = neo4j_datadir + 'post.csv'
user_file = neo4j_datadir + 'user.csv'

# 读取原始 CSV 文件
comment_df = pd.read_csv(comment_file)
post_df = pd.read_csv(post_file)
user_df = pd.read_csv(user_file)

# 更新列：添加 ':LABEL'
comment_df.insert(0, ':LABEL', 'Comment')
post_df.insert(0, ':LABEL', 'Post')
user_df.insert(0, ':LABEL', 'User')

# 将更新后的数据保存回原文件
comment_df.to_csv(comment_file, index=False)
post_df.to_csv(post_file, index=False)
user_df.to_csv(user_file, index=False)

print("Labels added successfully!")


Labels added successfully!


6.1 均匀随机分配更细致的标签 
经过上面代码为数据分配标签后，我想更细致的分配标签，对于每个csv文件，以post为例均匀概率分配100个不同的标签，标签格式为 post-1, post-2,... post-100,其他文件也类似重新分配标签

In [ ]:
import pandas as pd
import numpy as np
import os

# 文件路径
neo4j_datadir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/'

# 配置：每个文件对应的标签数量
# 例如 comment 生成 50 个标签，post 生成 100 个标签，user 生成 20 个标签
label_counts = {
    'comment': 10,
    'post':    10,
    'user':    5,
}

# 准备文件名映射
files = {
    'comment': 'comment.csv',
    'post':    'post.csv',
    'user':    'user.csv',
}

for prefix, filename in files.items():
    # 读取
    path = os.path.join(neo4j_datadir, filename)
    df = pd.read_csv(path)
    
    # 取出本文件所需的标签数量
    n_labels = label_counts.get(prefix, 100)  # 默认为100
    # 构造标签列表： prefix-1 ... prefix-n_labels
    labels = [f"{prefix}-{i}" for i in range(1, n_labels+1)]
    
    # 随机均匀地分配给每一行
    df[':LABEL'] = np.random.choice(labels, size=len(df))
    
    # 保存
    df.to_csv(path, index=False)
    print(f"{filename} → 使用 {n_labels} 个标签，处理行数 {len(df)}。")


### 先修改配置文件，使用一下问题初始化数据库 testx
dbms.default_database=test1
neo4j restart
cypher-shell -a bolt://localhost:7688 -u neo4j -p neo4jws
neo4j stop
### 下面是在neo4j中导入数据的命令：
neo4j-admin import --nodes=/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/comment.csv --nodes=/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/post.csv --nodes=/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/user.csv --relationships=/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/have_relationship.csv --relationships=/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/public_relationship.csv --relationships=/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/view_relationship.csv --database=test3

neo4j-admin import --nodes=/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/comment.csv --nodes=/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/post.csv --nodes=/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/user.csv --relationships=/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/have_relationship.csv --relationships=/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/public_relationship.csv --relationships=/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/neo4j_data/view_relationship.csv --database=neo4j

### 若导入失败，要删除已生成的数据库文件，运行如下命令：
rm -rf /home/wangshuo/software/neo4j-community-4.4.0/data/databases/parler30w


### 导入数据图成功后
我user的标签全都是user-1,user-2.... post的标签是post-1,post-2.....,comment的标签为comment-1，comment-2......, 我已有的图上，先匹配用user-i --public-> comment-j <--have-- post-k 模式，对于每个匹配到的模式添加user-i 到 post - k 的边，此边为--view-->边，   其中不同的user标签有5种，post有10种。comment有10种

In [1]:
import os
import itertools
from neo4j import GraphDatabase
from tqdm import tqdm

# 1. Neo4j 连接配置
URI      = "bolt://localhost:7688"
USERNAME = "neo4j"
PASSWORD = "neo4jws"  # 替换成你的密码

# 2. 准备标签列表
user_labels    = [f"user-{i}"    for i in range(1, 6)]
comment_labels = [f"comment-{j}" for j in range(1, 11)]
post_labels    = [f"post-{k}"    for k in range(1, 11)]

# 3. 建立 Neo4j 驱动
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

# 4. 开始会话并执行
total = len(user_labels) * len(comment_labels) * len(post_labels)
with driver.session() as session:
    for u, c, p in tqdm(itertools.product(user_labels, comment_labels, post_labels),
                       total=total,
                       desc="Adding view edges"):
        cypher = f"""
        MATCH (u:`{u}`)-[:public]->(c:`{c}`)<-[:have]-(p:`{p}`)
        MERGE (u)-[:view]->(p);
        """
        # 只执行，不返回结果
        session.run(cypher)

driver.close()
print("所有 view 边添加完成！")


Adding view edges: 100%|██████████| 500/500 [00:19<00:00, 25.24it/s]

所有 view 边添加完成！


# 保证数据完整性

6.节点数据：comment.csv、post.csv、user.csv
关系CSV：have_relationship.csv、public_relationship.csv、view_relationship.csv
利用上面数据在neo4j中通过 neo4j-admin import 的方式建立图:
1. 对users_processed、posts_processed、comments_processed进行数据清理，将换行符去掉

7. 接下来补全数据集保证所有的post和comment都有用户，所有的comment都有post

In [ ]:
import pandas as pd

# 输入文件路径
neo4j_datadir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/'
comments_file = neo4j_datadir + 'comment.csv'
posts_file = neo4j_datadir + 'post.csv'
users_file = neo4j_datadir + 'user.csv'

# 输出文件路径
lost_user_file = neo4j_datadir + 'lost/lost_user.csv'
lost_post_file = neo4j_datadir + 'lost/lost_post.csv'
lost_comment_user_file = neo4j_datadir + 'lost/lost_comment_user.csv'

# 读取 CSV 文件
posts_df = pd.read_csv(posts_file)
comments_df = pd.read_csv(comments_file)
users_df = pd.read_csv(users_file)

# 获取所有 posts 的 ID 和 users 的 ID
post_ids = set(posts_df['id:ID'])
user_ids = set(users_df['id:ID'])

# 获取 comments 中的 user_id 和 post_id
comments_user_ids = set(comments_df['creator:ID'])
comments_post_ids = set(comments_df['post:ID'])

# 检查是否每个 post 对应一个 user
lost_user_ids = posts_df[~posts_df['creator:ID'].isin(user_ids)]['creator:ID']

# 检查是否每个 comment 对应一个 post
lost_post_ids = comments_df[~comments_df['post:ID'].isin(post_ids)]['post:ID']

# 检查是否每个 comment 对应一个 user
lost_comment_user_ids = comments_df[~comments_df['creator:ID'].isin(user_ids)]['creator:ID']

# 保存缺失的用户和帖子 ID 到 CSV 文件
lost_user_ids.to_csv(lost_user_file, index=False, header=['lost_user_id'])
lost_post_ids.to_csv(lost_post_file, index=False, header=['lost_post_id'])
lost_comment_user_ids.to_csv(lost_comment_user_file, index=False, header=['lost_comment_user_id'])

# 输出检查结果
print(f"Lost user IDs saved to {lost_user_file}")
print(len(lost_user_ids))

print(f"Lost post IDs saved to {lost_post_file}")
print(len(lost_post_ids))

print(f"Lost comment user IDs saved to {lost_comment_user_file}")
print(len(lost_comment_user_ids))


In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 输入文件路径
lost_user_file = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/workload3/lost/lost_user.csv'
valid_users_dir = '/home/wangshuo/resource/datasets/parler/csv_data/user/all_users'
output_file = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/workload3/lost/find_users.csv'

# 读取 lost_user.csv，获取 lost_user 的所有 ID
lost_user_df = pd.read_csv(lost_user_file)
lost_user_ids = set(lost_user_df['lost_user_id'])

# 查找 valid_users 目录下的所有 CSV 文件
valid_user_files = [f for f in os.listdir(valid_users_dir) if f.endswith('.csv')]

# 存储找到的用户元组
found_users_data = []

# 使用 tqdm 给循环加上进度条
for valid_user_file in tqdm(valid_user_files, desc="Processing valid user files", unit="file"):
    valid_user_path = os.path.join(valid_users_dir, valid_user_file)
    valid_user_df = pd.read_csv(valid_user_path)

    # 找到 lost_user_ids 中在 valid_user_ids 中存在的部分
    matched_users = valid_user_df[valid_user_df['id'].isin(lost_user_ids)]

    # 将找到的用户数据添加到列表中
    found_users_data.append(matched_users)

# 合并所有找到的用户数据
if found_users_data:
    found_users_df = pd.concat(found_users_data, ignore_index=True)

    # 保存到 CSV 文件
    found_users_df.to_csv(output_file, index=False)
    print(f"Found users saved to {output_file}")
else:
    print("No users found.")

# 判断是否所有 lost_user_id 都能找到
found_users = set(found_users_df['id']) if found_users_data else set()
if len(found_users) == len(lost_user_ids):
    print("All users found: ok")
else:
    print("Some users not found: not")
    missing_users = lost_user_ids - found_users
    print(f"Missing {len(missing_users)} users: {missing_users}")


# 8.对CSV列进行选择保存

In [ ]:
import pandas as pd

# 输入文件路径
users_file = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/workload3/users_trimmed1.csv'

# 读取 CSV 文件
users_df = pd.read_csv(users_file)

# 为 :LABEL 列的所有元组赋值为 User
users_df[':LABEL'] = 'User'

# 保存修改后的文件
users_df.to_csv(users_file, index=False)

print(f"Updated :LABEL column in {users_file} to 'User'")


In [ ]:
import pandas as pd

# 文件路径
input_file = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/workload3/users_trimmed1.csv'
output_file = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/workload3/users_trimmed1.csv'

# 读取 CSV 文件
users_df = pd.read_csv(input_file)

# 删除指定的列
columns_to_remove = ['creator_x', 'creator_y']
users_df = users_df.drop(columns=columns_to_remove, errors='ignore')  # 使用 errors='ignore' 防止列不存在时报错

# 保存修改后的数据到新文件
users_df.to_csv(output_file, index=False)

print(f"Removed columns {columns_to_remove} and saved to {output_file}")


In [ ]:
import pandas as pd

# 文件路径
input_file = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/workload3/lost/find_users_processed_cleaned.csv'
output_file = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/workload3/lost/find_users_processed_cleaned.csv'

# 读取 CSV 文件
users_df = pd.read_csv(input_file)

# 添加 upNum 列，默认值为 0
users_df['ucNum'] = 0

# 保存修改后的数据到新文件
users_df.to_csv(output_file, index=False)

print(f"Added 'upNum' column with default value 0 and saved to {output_file}")


In [ ]:
print(len(lost_user_ids))
import pandas as pd

# 读取现有的 users_processed_cleaned.csv 和 find_users_processed_cleaned.csv
users_df = pd.read_csv('/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload3/users_trimmed1.csv')
find_users_df = pd.read_csv('/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload3/lost/find_users_processed_cleaned.csv')

# 将 find_users_df 的数据添加到 users_df 中
combined_df = pd.concat([users_df, find_users_df], ignore_index=True)

# 保存合并后的 DataFrame 到新的 CSV 文件
combined_df.to_csv('/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload3/lost/combined_users_processed.csv', index=False)

print("Users have been successfully combined and saved to 'combined_users_processed.csv'.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 文件路径
neo4j_datadir = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/30W_valid_user/neo4j/'
file_path = neo4j_datadir + 'post.csv'

# 读取文件
print("读取 posts_test_large.csv 文件...")
posts_df = pd.read_csv(file_path)

# 计算 pcNum=0 的元组数量
pcNum_zero_count = (posts_df['pcNum'] == 0).sum()
print(f"pcNum=0 的元组数量: {pcNum_zero_count}")
print(f"post 的元组数量: {len(posts_df)}")

# 定义 pn 数组
pn = [1,2, 4, 8, 16, 32, 64, 128, 254]

# 计算并输出 pcNum > pn[i] 的 post 数量
for i, threshold in enumerate(pn):
    count = (posts_df['pcNum'] >= threshold).sum()
    print(f"pcNum >= {threshold} 的 post 数量: {count}")

# 绘制 pcNum 大小曲线
plt.figure(figsize=(10, 6))
plt.plot(posts_df['pcNum'], label='pcNum')
plt.xlabel('Index (i)')
plt.ylabel('pcNum')
plt.title('pcNum Value Curve')
plt.legend()
plt.grid(True)
plt.show()

# 9.接下来裁剪数据，选取pcNum >=4 的post， 然后根据这些post裁剪comment和user：
comments_processed_cleaned.csv，posts_processed_cleaned.csv，users_processed_cleaned.csv
是保存上面comment，post,user的文件，其中已知comments文件中元组通过外键post和posts文件的id：ID相连接，posts 和comments文件中元组通过外键creator与users的id：ID连接，
接下来裁剪数据，选取pcNum >=4 的post， 然后根据这些post裁剪comment和user--裁剪后的post集合对users,和commnets进行裁剪，保留和post有外键关联的comment和user，输出裁剪前后的post，user,comment的数量

In [ ]:
import pandas as pd

# 文件路径
base_path = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload3/'
comments_file = base_path + 'comments_processed_cleaned.csv'
posts_file = base_path + 'posts_processed_cleaned.csv'  # 使用处理后的 posts 文件
users_file = base_path + 'users_processed_cleaned.csv'


# 读取文件
print("读取文件...")
comments_df = pd.read_csv(comments_file)
posts_df = pd.read_csv(posts_file)
users_df = pd.read_csv(users_file)


# 输出裁剪前的数据量
print("裁剪前的数据量：")
print(f"  Posts: {len(posts_df)}")
print(f"  Comments: {len(comments_df)}")
print(f"  Users: {len(users_df)}")

# 裁剪 posts 数据
print("裁剪 posts 数据...")
filtered_posts_df = posts_df[posts_df['pcNum'] >= 4]
filtered_post_ids = filtered_posts_df['id:ID'].tolist()
print(f"  裁剪后 Posts: {len(filtered_posts_df)}")


# 裁剪 comments 数据
print("裁剪 comments 数据...")
filtered_comments_df = comments_df[comments_df['post'].isin(filtered_post_ids)]
print(f"  裁剪后 Comments: {len(filtered_comments_df)}")

# 裁剪 users 数据
print("裁剪 users 数据...")
filtered_user_ids_from_posts = filtered_posts_df['creator'].tolist()
filtered_user_ids_from_comments = filtered_comments_df['creator'].tolist()
filtered_user_ids = list(set(filtered_user_ids_from_posts + filtered_user_ids_from_comments))

filtered_users_df = users_df[users_df['id:ID'].isin(filtered_user_ids)]

print(f"  裁剪后 Users: {len(filtered_users_df)}")

# 输出裁剪后的数据量
print("裁剪后的数据量：")
print(f"  Posts: {len(filtered_posts_df)}")
print(f"  Comments: {len(filtered_comments_df)}")
print(f"  Users: {len(filtered_users_df)}")

# 保存裁剪后的数据（可选）
output_base_path =  '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload3/trimmed/'
filtered_posts_df.to_csv(output_base_path + 'posts_trimmed.csv', index=False)
filtered_comments_df.to_csv(output_base_path + 'comments_trimmed.csv', index=False)
filtered_users_df.to_csv(output_base_path + 'users_trimmed.csv', index=False)
print(f"裁剪后的文件保存到 {output_base_path} ")

10. users_trimmed.csv和comments_trimmed.csv元组之间做内连接，
也就是只保存comment的creator属性能够在users的id列表中找到的comment，和在user的id能够在comments 的creator列表中找到的user

In [ ]:
import pandas as pd

# 文件路径
base_path = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload3/trimmed/'
users_file = base_path + 'users_trimmed.csv'
comments_file = base_path + 'comments_trimmed.csv'

# 读取文件
print("读取文件...")
users_df = pd.read_csv(users_file)
comments_df = pd.read_csv(comments_file)

# 输出连接前的数据量
print("连接前的数据量：")
print(f"  Users: {len(users_df)}")
print(f"  Comments: {len(comments_df)}")


# 执行内连接操作
print("执行内连接操作...")
# 内连接 comments，保留 creator 在 users id 中的 comment
filtered_comments_df = comments_df[comments_df['creator'].isin(users_df['id:ID'])]

# 内连接 users,保留 users id 在 comments creator 中的 user
filtered_users_df = users_df[users_df['id:ID'].isin(filtered_comments_df['creator'])]

print("连接后的数据量:")
print(f"  Users: {len(filtered_users_df)}")
print(f"  Comments: {len(filtered_comments_df)}")



# 保存连接后的数据 (可选)
output_base_path =  '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload3/trimmed/'
filtered_comments_df.to_csv(output_base_path + 'comments_inner_join.csv', index=False)
filtered_users_df.to_csv(output_base_path + 'users_inner_join.csv', index=False)

print(f"内连接后的文件保存到 {output_base_path}")

In [ ]:
import pandas as pd

# 文件路径
base_path = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/workload3/trimmed/'
users_file = base_path + 'users_inner_join.csv'
comments_file = base_path + 'comments_inner_join.csv'

# 读取文件
print("读取文件...")
users_df = pd.read_csv(users_file)
comments_df = pd.read_csv(comments_file)

# 验证 comments 中的 creator 是否都在 users 中
print("验证 comments 中的 creator 是否都在 users 中...")
comments_creators = comments_df['creator'].tolist()
users_ids = users_df['id:ID'].tolist()

all_creators_in_users = all(creator in users_ids for creator in comments_creators)

if all_creators_in_users:
    print("  ✅ comments 中所有 creator 都在 users 中")
else:
    print("  ❌ comments 中存在 creator 不在 users 中")
    
# 验证 users 中的 id 是否在 comments 中存在对应的 creator
print("验证 users 中的 id 是否在 comments 中存在对应的 creator...")
users_ids = users_df['id:ID'].tolist()
comments_creators = comments_df['creator'].tolist()

all_users_have_comments = all(user_id in comments_creators for user_id in users_ids)

if all_users_have_comments:
     print("  ✅ users 中所有 id 都在 comments 中有对应的 creator")
else:
     print("  ❌ users 中存在 id 在 comments 中没有对应的 creator")

# 验证 users 中的 id 是否在 comments 中存在至少一个对应的 creator，同时输出没有对应评论的用户
print("验证 users 中的 id 是否在 comments 中存在至少一个对应的 creator，并输出没有对应评论的用户")

users_with_no_comments = []
for user_id in users_ids:
    if user_id not in comments_creators:
        users_with_no_comments.append(user_id)

if len(users_with_no_comments) == 0:
    print(" ✅ users 中所有 id 都在 comments 中有对应的 creator (至少一个)")
else:
    print(" ❌ users 中存在 id 在 comments 中没有对应的 creator (至少一个)")
    print("   这些用户没有对应的评论:", users_with_no_comments)

In [ ]:
import pandas as pd

# 文件路径
post_test_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/post_test.csv'
posts_test_large_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/posts_test_large.csv'

# 读取文件
print("读取 post_test.csv 文件...")
post_test_df = pd.read_csv(post_test_file)
print("读取 posts_test_large.csv 文件...")
posts_test_large_df = pd.read_csv(posts_test_large_file)

# 验证 post_test.csv 是否是 posts_test_large.csv 的真子集
print("验证中...")
post_test_ids = set(post_test_df['id'])
posts_test_large_ids = set(posts_test_large_df['id'])

# 找到不在 posts_test_large.csv 的 id
not_in_large = post_test_ids - posts_test_large_ids
not_in_large_count = len(not_in_large)

if not_in_large_count > 0:
    print(f"有 {not_in_large_count} 个元组在 post_test.csv 中，但不在 posts_test_large.csv 中。")
    print("以下是不在的 id 示例：")
    print(list(not_in_large)[:10])  # 打印前 10 个不在的 id
else:
    print("post_test.csv 的所有元组都在 posts_test_large.csv 中。")

# 如果需要保存不在的元组到文件
if not_in_large_count > 0:
    not_in_large_df = post_test_df[post_test_df['id'].isin(not_in_large)]
    not_in_large_df.to_csv('/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/not_in_large_posts.csv', index=False)
    print("不在 posts_test_large.csv 中的元组已保存到 not_in_large_posts.csv 文件中。")


In [ ]:
import pandas as pd

# 文件路径
not_in_large_posts_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/not_in_large_posts.csv'
posts_test_large_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/posts_test_large.csv'
users_test_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/users_test.csv'

# 读取文件
print("读取 not_in_large_posts.csv 文件...")
not_in_large_posts_df = pd.read_csv(not_in_large_posts_file)

print("读取 posts_test_large.csv 文件...")
posts_test_large_df = pd.read_csv(posts_test_large_file)

print("读取 users_test.csv 文件...")
users_test_df = pd.read_csv(users_test_file)

# 将 not_in_large_posts 追加到 posts_test_large
print("将 not_in_large_posts 追加到 posts_test_large.csv...")
updated_posts_test_large_df = pd.concat([posts_test_large_df, not_in_large_posts_df], ignore_index=True)

# 保存更新后的 posts_test_large.csv
updated_posts_test_large_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/posts_test_large_updated.csv'
updated_posts_test_large_df.to_csv(updated_posts_test_large_file, index=False)
print(f"更新后的 posts_test_large.csv 已保存到 {updated_posts_test_large_file}")

# 检查 not_in_large_posts 中的 creator 是否在 users_test.csv 中
print("检查 not_in_large_posts 的 creator 是否在 users_test.csv 中...")
not_in_large_creators = set(not_in_large_posts_df['creator'])
users_test_ids = set(users_test_df['id'])

# 找到不在 users_test.csv 中的 creator
missing_creators = not_in_large_creators - users_test_ids
missing_count = len(missing_creators)

if missing_count > 0:
    print(f"有 {missing_count} 个 creator 在 users_test.csv 中找不到。")
    print("以下是一些示例缺失的 creator：")
    print(list(missing_creators)[:10])  # 示例打印前 10 个缺失的 creator

    # 保存缺失的 creator 到文件
    missing_creators_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/missing_creators.csv'
    pd.DataFrame({'missing_creator': list(missing_creators)}).to_csv(missing_creators_file, index=False)
    print(f"缺失的 creator 已保存到 {missing_creators_file}")
else:
    print("所有的 creator 都在 users_test.csv 中找到。")


In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 文件路径
filtered_comments_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/comments_test_large.csv'
posts_dir = '/home/wangshuo/resource/DataSets/parler/csv_data/pc/posts/valid_posts'
posts_test_large_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/posts_test_large.csv'
updated_filtered_comments_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/comments_test_large.csv'

# 读取 filtered_comments.csv
print("读取 filtered_comments.csv 文件...")
filtered_comments_df = pd.read_csv(filtered_comments_file)

# 初始化匹配后的评论和帖子数据
matched_comments_list = []
matched_posts_list = []

# 扫描 posts 文件夹
post_files = [f for f in os.listdir(posts_dir) if f.endswith('.csv')]

print("开始处理 posts 文件...")
for post_file in tqdm(post_files, desc="扫描 posts 文件"):
    # 读取 posts 文件
    post_file_path = os.path.join(posts_dir, post_file)
    posts_df = pd.read_csv(post_file_path)
    
    # 筛选 filtered_comments 中有对应 post 的评论
    matched_comments = filtered_comments_df[filtered_comments_df['post'].isin(posts_df['id'])]
    matched_comments_list.append(matched_comments)
    
    # 筛选 posts 中有对应评论的帖子
    matched_posts = posts_df[posts_df['id'].isin(filtered_comments_df['post'])]
    matched_posts_list.append(matched_posts)

# 合并所有匹配的评论和帖子数据
final_filtered_comments = pd.concat(matched_comments_list, ignore_index=True)
final_posts = pd.concat(matched_posts_list, ignore_index=True)

# 删除 filtered_comments 中无匹配 post 的评论
print("删除找不到对应 post 的评论...")
updated_filtered_comments = final_filtered_comments.copy()



In [ ]:
# 按 comment 字段降序排序
final_filtered_comments_sorted = final_filtered_comments.sort_values(by='comments', ascending=False)

# 保存 posts_test_large.csv（帖子数据）
print(f"保存帖子数据到 {posts_test_large_file}...")
final_posts.to_csv(posts_test_large_file, index=False)

# 更新 filtered_comments.csv（评论数据）
print(f"更新评论数据到 {updated_filtered_comments_file}...")
updated_filtered_comments.to_csv(updated_filtered_comments_file, index=False)

# 打印示例数据
print("更新后的评论数据示例：")
print(updated_filtered_comments.head())

print("保存的帖子数据示例：")
print(final_posts.head())


In [ ]:
print(len(updated_filtered_comments))
print(len(final_posts))